# Day 1-01｜球場 Keypoint 配對與 Homography 標定
> Python 籃球運動資料分析課程  
> 本單元建立相機畫面與 BEV 球場之間的對應點，並計算 Homography 投影矩陣。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 在 BEV 球場圖選擇具名 keypoint。
- 在相機畫面點選同一個實體位置，建立 keypoint pair。
- 使用至少四組 pair 計算 Homography。
- 直接點選相機畫面任意位置，觀察矩陣如何投影到 BEV。
- 將相同概念延伸到 player footpoint 與 detection box 應用。

## 完成產出
- `assets/results/d1_01_bev_keypoint_reference.png`
- `assets/results/d1_01_homography_pairs.png`
- `assets/results/d1_01_projected_footpoint.png`


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 初始化 Colab 或本機課程環境。
2. 產生 BEV keypoint 參考圖。
3. 使用 pair 標定工具建立 BEV/camera 對應點。
4. 計算 Homography 並檢查配對結果。
5. 以互動工具點選任意畫面位置，觀察投影到 BEV 的結果。
6. 將投影概念延伸到 player footpoint。


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


In [ ]:
from src.cv_utils import save_image_rgb
from src.geometry_utils import render_bev_court
from src.yolo_utils import (
    COURT_KEYPOINT_DISPLAY_NAMES,
    bev_court_bounds,
    court_vertices_bev_in_bounds,
)

CAMERA_IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
BEV_IMAGE_PATH = COURSE_ROOT / "assets" / "results" / "d1_01_bev_keypoint_reference.png"

bev = render_bev_court(BEV_SPEC_PATH)
save_image_rgb(BEV_IMAGE_PATH, bev)

bev_points_all = court_vertices_bev_in_bounds(bev_court_bounds(BEV_SPEC_PATH))
bev_keypoints = [
    {"name": name, "xy": [float(x), float(y)]}
    for name, (x, y) in zip(COURT_KEYPOINT_DISPLAY_NAMES, bev_points_all)
]

print("camera image:", CAMERA_IMAGE_PATH)
print("BEV keypoint reference:", BEV_IMAGE_PATH)
print("keypoints:", len(bev_keypoints))


In [ ]:
from src.notebook_widgets import show_court_keypoint_pair_tool

show_court_keypoint_pair_tool(
    bev_image_path=BEV_IMAGE_PATH,
    camera_image_path=CAMERA_IMAGE_PATH,
    keypoints=bev_keypoints,
    canvas_width=1000,
    title="BEV / Camera Keypoint Pairing",
)


## Step 1｜貼上工具輸出的配對 JSON
請將工具輸出的完整 JSON 直接覆蓋到下一格的 `PAIR_DATA_JSON`。後續計算會優先使用 JSON 內原始輸出的 `camera_xy` 與 `bev_xy`，不會再次改寫配對座標。


In [ ]:

PAIR_DATA_JSON = r"""
# TODO: 在這裡貼上從工具中匯出的 JSON 內容
"""

## Step 2｜理解 Homography 在做什麼
當相機畫面中的點與 BEV 球場上的點都落在**同一個平面**上時，可以用一個 $3 \times 3$ 矩陣 $H$ 描述兩者的對應關係：

$$
\lambda
\begin{bmatrix}
u' \\
v' \\
1
\end{bmatrix}
=
H
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
$$

- $(u, v)$：相機畫面中的點
- $(u', v')$：BEV 球場中的對應點
- $\lambda$：齊次座標的比例常數

由於 $H$ 有 8 個自由度，所以至少需要 **4 組對應點** 才能求解。實務上通常會多貼幾組點，讓矩陣更穩定，也更容易檢查哪個點貼歪了。


In [ ]:
import json
import pandas as pd

pair_data = json.loads(PAIR_DATA_JSON)
if "pairs" not in pair_data or not isinstance(pair_data["pairs"], list):
    raise ValueError("貼入的 JSON 內容中找不到 'pairs' 欄位或其格式不正確。")

bev_lookup = {item["name"]: item["xy"] for item in bev_keypoints}

pairs = []
for raw_pair in pair_data["pairs"]:
    keypoint_name = str(raw_pair["keypoint_name"])
    if "bev_xy" in raw_pair:
        bev_xy = [float(v) for v in raw_pair["bev_xy"]]
    else:
        if keypoint_name not in bev_lookup:
            raise KeyError(f"找不到 BEV keypoint: {keypoint_name}")
        bev_xy = [float(v) for v in bev_lookup[keypoint_name]]
    pairs.append(
        {
            "keypoint_name": keypoint_name,
            "camera_xy": [float(v) for v in raw_pair["camera_xy"]],
            "bev_xy": bev_xy,
        }
    )

if len(pairs) < 4:
    raise ValueError("Homography 至少需要四組 BEV/camera pair。")

pair_rows = []
for idx, pair in enumerate(pairs):
    pair_rows.append(
        {
            "index": idx,
            "keypoint_name": pair["keypoint_name"],
            "camera_x": float(pair["camera_xy"][0]),
            "camera_y": float(pair["camera_xy"][1]),
            "bev_x": float(pair["bev_xy"][0]),
            "bev_y": float(pair["bev_xy"][1]),
        }
    )

print("pair count:", len(pairs))
pd.DataFrame(pair_rows)


## Step 3｜整理成矩陣輸入格式
OpenCV 的 `cv2.findHomography()` 需要兩組對應座標陣列：

- `camera_points_np`：相機畫面點
- `bev_points_np`：BEV 球場點

這一步只是把剛剛的 JSON 轉成數值矩陣，方便後面求解與投影。


In [ ]:
import numpy as np

camera_points = [pair["camera_xy"] for pair in pairs]
bev_points = [pair["bev_xy"] for pair in pairs]
point_names = [pair["keypoint_name"] for pair in pairs]

camera_points_np = np.asarray(camera_points, dtype=np.float32)
bev_points_np = np.asarray(bev_points, dtype=np.float32)
if len(camera_points_np) < 4:
    raise ValueError("Homography 至少需要 4 組對應點。")

print("camera_points_np shape:", camera_points_np.shape)
print("bev_points_np shape:", bev_points_np.shape)
print("camera_points_np:\n", camera_points_np)
print("bev_points_np:\n", bev_points_np)


## Step 4｜求解 Homography 矩陣，並檢查 reprojection error
接著用 `cv2.findHomography()` 求出矩陣 $H$。算完之後，不要只看矩陣本身，還要檢查它把原本的相機點投回 BEV 時，誤差有多大。

對每一組配對點，我們會計算：

$$
e_i = \left\| \hat{\mathbf{p}}_i' - \mathbf{p}_i' \right\|_2
$$

- $\mathbf{p}_i'$：你手動貼的 BEV 真值
- $\hat{\mathbf{p}}_i'$：把相機點經過 $H$ 投影後得到的 BEV 預測點
- $e_i$：第 $i$ 個點的重投影誤差（pixel）

如果某個點的誤差特別大，通常代表那一組 pair 點錯、點位順序不一致，或貼到不是同一個實體位置。


In [ ]:
import cv2

H, _ = cv2.findHomography(camera_points_np, bev_points_np, method=0)
if H is None:
    raise RuntimeError("cv2.findHomography 失敗，請確認點位順序與座標是否正確。")

projected_bev_points = cv2.perspectiveTransform(
    camera_points_np.reshape(-1, 1, 2),
    H,
).reshape(-1, 2)
reprojection_errors = np.linalg.norm(projected_bev_points - bev_points_np, axis=1)
error_rows = []
for idx, (name, gt, pred, err) in enumerate(zip(point_names, bev_points_np, projected_bev_points, reprojection_errors)):
    error_rows.append(
        {
            "index": idx,
            "keypoint_name": name,
            "gt_bev_x": float(gt[0]),
            "gt_bev_y": float(gt[1]),
            "pred_bev_x": float(pred[0]),
            "pred_bev_y": float(pred[1]),
            "reprojection_error_px": float(err),
        }
    )

print("Homography matrix H =")
print(np.round(H, 3))
print("mean reprojection error (px):", float(np.mean(reprojection_errors)))
print("max reprojection error (px):", float(np.max(reprojection_errors)))
pd.DataFrame(error_rows)


## Step 5｜把配對點與投影結果畫出來
最後做視覺檢查：

- 左圖：相機畫面上的手動配對點
- 右圖：BEV 上的真值點與投影後點

如果兩者很接近，表示矩陣合理；如果偏差很大，就應該回去檢查 JSON 裡的點位。


In [ ]:
from src.cv_utils import draw_points, read_image_rgb, show_image, side_by_side

frame = read_image_rgb(CAMERA_IMAGE_PATH)
bev = render_bev_court(BEV_SPEC_PATH)
frame_vis = draw_points(frame, camera_points, labels=point_names)
bev_vis = draw_points(bev, bev_points, labels=point_names, color=(40, 120, 255), radius=7)
bev_vis = draw_points(bev_vis, projected_bev_points, labels=[f"proj_{idx}" for idx in range(len(projected_bev_points))], color=(255, 99, 72), radius=5)
combo = side_by_side(frame_vis, bev_vis, max_width=2800)
show_image(combo, "Camera-BEV calibration pairs and reprojection", figsize=(18, 8))

out_path = COURSE_ROOT / "assets" / "results" / "d1_01_homography_pairs.png"
save_image_rgb(out_path, combo)
print("saved:", out_path)


## Step 6｜任意點投影觀察
請直接點選相機畫面任意位置。工具會使用上一步計算出的 Homography 矩陣，立即將該點投影到右側 BEV，協助觀察「同一矩陣」如何把相機座標映射到球場平面。


In [ ]:
from src.notebook_widgets import show_homography_projection_tool

show_homography_projection_tool(
    image_path=CAMERA_IMAGE_PATH,
    bev_image_path=BEV_IMAGE_PATH,
    homography_matrix=H.tolist(),
    canvas_width=1000,
    title="Click-to-BEV Projection",
)


## Step 7｜Player Footpoint 應用
Homography 描述的是同一平面上的投影關係。實務上會先從 player bbox 取底邊中心點，作為球員與地面接觸位置的近似值，再投影到 BEV。

如果 bbox 記成 $(x_1, y_1, x_2, y_2)$，那麼常用的 footpoint 近似是：

$$
\text{footpoint} = \left( \frac{x_1 + x_2}{2},\ y_2 \right)
$$

也就是 **底邊中心點**。這在球員站立、腳落在地板上的情境下，通常比 bbox 中心更接近真實場上位置。


In [ ]:
from src.cv_utils import bottom_center, draw_boxes
from src.geometry_utils import DEFAULT_DEMO_PLAYER_BBOX_XYXY

bbox = DEFAULT_DEMO_PLAYER_BBOX_XYXY
footpoint = bottom_center(bbox)
pts = np.asarray([footpoint], dtype=np.float32).reshape(-1, 1, 2)
bev_footpoint = cv2.perspectiveTransform(pts, H).reshape(-1, 2)[0]

frame_vis = draw_boxes(frame, [bbox], ["player_bbox"])
frame_vis = draw_points(frame_vis, [footpoint], ["footpoint"])
bev_vis = draw_points(bev, [bev_footpoint], ["projected_footpoint"])
combo = side_by_side(frame_vis, bev_vis)
show_image(combo, "Projected footpoint")

out_path = COURSE_ROOT / "assets" / "results" / "d1_01_projected_footpoint.png"
save_image_rgb(out_path, combo)
print("footpoint:", [round(v, 2) for v in footpoint])
print("BEV footpoint:", np.round(bev_footpoint, 2).tolist())
print("saved:", out_path)


## 本單元產出檔案

- `assets/results/d1_01_bev_keypoint_reference.png`：BEV 球場 keypoint 參考圖。
- `assets/results/d1_01_homography_pairs.png`：camera / BEV 對應點標定示意圖。
- `assets/results/d1_01_projected_footpoint.png`：player footpoint 投影到 BEV 的示意圖。
